# 💻 GUÍA PRÁCTICA AVANZADA: ARQUITECTURA MODEL CONTEXT PROTOCOL (MCP)
## Versión Corregida para Databricks — Paquetes JSON-RPC en Vivo y Swapping de Modelos (Google Gemini vs Nvidia Nemotron)

Este notebook implementa la arquitectura MCP de manera robusta y estable para clústeres de Databricks:
- **Transporte**: Exclusivamente `stdio` (el único viable en la nube de Databricks; SSE requiere un servidor local en tu PC).
- **Modelos**: Google Gemini (SDK oficial) o Nvidia Nemotron (vía OpenRouter). 
- **Visualización**: Tarjetas JSON-RPC con diseño neón que muestran cada mensaje del protocolo MCP en tiempo real.

---

## 📦 Paso 1: Instalación de Dependencias en el Clúster de Databricks
Instalamos todas las librerías directamente sobre el entorno del clúster y reiniciamos Python para que sean reconocidas de inmediato.

In [ ]:
%pip install mcp openai pandas scikit-learn numpy sqlalchemy pymysql python-dotenv google-generativeai

In [ ]:
# Reiniciar Python para que las nuevas librerías sean cargadas en memoria
%restart_python

---

## 🛠️ Paso 2: Importar Librerías
Importamos MCP, SDKs de IA, y utilidades de visualización. Usamos `sys.executable` para que Databricks levante el servidor MCP con el intérprete de Python correcto del clúster, sin depender de binarios externos como `uv`.

In [ ]:
import os
import sys
import json
import asyncio
from dotenv import load_dotenv
from openai import AsyncOpenAI
import google.generativeai as genai
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from IPython.display import display, HTML

print(f"✅ Librerías cargadas. Python activo: {sys.executable}")

---

## 🔑 Paso 3: Carga y Verificación de Credenciales
Cargamos las variables privadas del archivo `.env` y verificamos el estado de cada clave disponible.

In [ ]:
load_dotenv(override=True)

gemini_key = os.getenv("GEMINI_API_KEY", "")
openrouter_key = os.getenv("OPENROUTER_API_KEY", "")

print("📋 Estado de Credenciales:")
print(f"   ➔ GEMINI_API_KEY    : {'✅ Configurada' if gemini_key else '❌ Vacía'}")
print(f"   ➔ OPENROUTER_API_KEY: {'✅ Configurada' if openrouter_key else '❌ Vacía'}")

---

## 🌐 Paso 4: Selección de Modelo de IA
Elige con qué cerebro de IA quieres orquestar la tarea. Podrás ver la diferencia en cómo cada SDK interactúa con exactamente el mismo servidor MCP local, sin cambiar una sola línea de código del servidor.

In [ ]:
print("=======================================================================")
print(" SELECCIÓN DE MODELO E INTEGRACIÓN:")
print("-----------------------------------------------------------------------")
print(" [1] -> Google Gemini 1.5 Flash   (AI Studio - Estable/Por Defecto)")
print(" [2] -> Google Gemini 2.5 Flash   (AI Studio - Nueva generación)")
print(" [3] -> Google Gemma 2 27B IT     (AI Studio - Modelo abierto ligero)")
print(" [4] -> Nvidia Nemotron-3 120B    (OpenRouter - Gratis)")
print(" [5] -> Modelo Personalizado      (Escribe el nombre de tu modelo de AI Studio)")
print("=======================================================================\n")

opcion_modelo = input("🧠 Selecciona tu modelo (1-5, ENTER = [1] por defecto): ").strip()

if opcion_modelo == "2":
    ia_proveedor = "gemini"
    modelo_activo = "gemini-2.5-flash"
elif opcion_modelo == "3":
    ia_proveedor = "gemini"
    modelo_activo = "gemma-2-27b-it"
elif opcion_modelo == "4":
    ia_proveedor = "openrouter"
    modelo_activo = "nvidia/nemotron-3-super-120b-a12b:free"
elif opcion_modelo == "5":
    ia_proveedor = input("🔌 Elige proveedor ('gemini' o 'openrouter'): ").strip().lower()
    modelo_activo = input("🧠 Escribe el nombre exacto de la IA (ej. 'gemma-2-9b-it'): ").strip()
else:
    ia_proveedor = "gemini"
    modelo_activo = "gemini-1.5-flash"

# Validaciones de credenciales
if ia_proveedor == "gemini" and not gemini_key:
    raise ValueError("❌ Elegiste Google AI Studio pero GEMINI_API_KEY está vacía en tu .env.")
elif ia_proveedor == "openrouter" and not openrouter_key:
    raise ValueError("❌ Elegiste OpenRouter pero OPENROUTER_API_KEY está vacía en tu .env.")

print(f"\n✅ Proveedor: {ia_proveedor.upper()} | Modelo Activo: {modelo_activo}")

---

## ⚙️ Paso 5: Configurar Parámetros del Servidor MCP
Usamos `sys.executable` para garantizar que Databricks ejecute el Servidor MCP con el intérprete del clúster activo. Esto elimina definitivamente el error `FileNotFoundError: 'uv'`.

In [ ]:
server_params = StdioServerParameters(
    command=sys.executable,          # El intérprete Python activo del clúster
    args=["mcp_server.py"],          # El servidor MCP de nuestro proyecto
    env=os.environ.copy()            # Heredar las credenciales del .env
)

print(f"✅ Servidor MCP configurado con: {sys.executable} mcp_server.py")

---

## 🎨 Paso 6: Sniffer Visual de Paquetes JSON-RPC
Esta función dibuja tarjetas de diseño neón/glassmorphism que interceptan y muestran en tiempo real exactamente qué mensajes JSON-RPC viajan por el canal de comunicación del protocolo MCP:
- 🟦 **Azul**: Peticiones que envía el Cliente MCP.
- 🟩 **Verde**: Respuestas que retorna el Servidor MCP.
- 🟨 **Naranja**: Decisiones de acción dictadas por el modelo de IA.

In [ ]:
def mostrar_log_json_rpc(direccion, metodo, payload):
    colores = {
        "CLIENTE_ENVIANDO":    ("📤 CLIENTE MCP → SOLICITANDO ACCIÓN (OUTBOX)",  "#00c3ff", "rgba(0,195,255,0.05)"),
        "SERVIDOR_RESPONDIENDO": ("📥 SERVIDOR MCP → RETORNANDO RESPUESTA (INBOX)", "#00ff88", "rgba(0,255,136,0.05)"),
        "IA_DECIDIENDO":       ("🧠 INTELIGENCIA ARTIFICIAL → INVOCANDO HERRAMIENTA", "#ffaa00", "rgba(255,170,0,0.05)"),
    }
    titulo, borde, bg = colores.get(direccion, ("📋 LOG", "#888", "rgba(0,0,0,0.05)"))
    json_str = json.dumps(payload, indent=2, ensure_ascii=False)
    display(HTML(f"""
    <div style="border:2px solid {borde};background:{bg};padding:15px;border-radius:8px;
                margin:12px 0;font-family:Consolas,monospace;box-shadow:0 4px 15px rgba(0,0,0,.3);">
      <div style="font-weight:bold;color:{borde};margin-bottom:8px;font-size:1.1em;">{titulo}</div>
      <div style="font-size:.85em;color:#94a3b8;margin-bottom:6px;">Método: <code>{metodo}</code></div>
      <pre style="background:#0f172a;padding:10px;border-radius:5px;color:#38bdf8;
                  overflow-x:auto;border:1px solid #1e293b;margin:0;">{json_str}</pre>
    </div>"""))

print("✅ Sniffer visual JSON-RPC cargado.")

---

## 🧹 Paso 7: Función de Limpieza de Esquemas MCP para Compatibilidad con Gemini
El SDK oficial de Google Gemini NO acepta ciertos campos como `"default"` o `"$schema"` dentro de los esquemas de herramientas MCP. Esta función los elimina recursivamente antes de pasarlos al modelo, evitando el error `ValueError: Unknown field for Schema: default`.

In [ ]:
# Campos que el SDK de Gemini acepta oficialmente en su clase Schema
CAMPOS_SOPORTADOS_GEMINI = {"type", "format", "description", "nullable", "enum", "properties", "required", "items"}

# Mapeo de tipos estándar de JSON Schema a los nombres oficiales del Enum de Protobuf de Google
MAPEO_TIPOS_GEMINI = {
    "object": "OBJECT",
    "string": "STRING",
    "integer": "INTEGER",
    "number": "NUMBER",
    "boolean": "BOOLEAN",
    "array": "ARRAY"
}

def limpiar_schema_para_gemini(schema: dict) -> dict:
    """Conserva únicamente los campos soportados por el SDK de Google Gemini y convierte los tipos a mayúsculas."""
    if not isinstance(schema, dict):
        return schema
    
    nuevo_schema = {}
    for k, v in schema.items():
        if k in CAMPOS_SOPORTADOS_GEMINI:
            if k == "type" and isinstance(v, str):
                nuevo_schema[k] = MAPEO_TIPOS_GEMINI.get(v.lower(), v)
            elif k == "properties" and isinstance(v, dict):
                nuevo_schema[k] = {pk: limpiar_schema_para_gemini(pv) for pk, pv in v.items()}
            elif k == "items" and isinstance(v, dict):
                nuevo_schema[k] = limpiar_schema_para_gemini(v)
            else:
                nuevo_schema[k] = v
    return nuevo_schema

print("✅ Función de limpieza de esquemas cargada.")

---

## 💬 Paso 8: Configuración Interactiva del Prompt
Presiona Enter para usar la instrucción predeterminada, o escribe tu propia consulta personalizada para el modelo de IA.

In [ ]:
prompt_predeterminado = (
    "Extrae 100 transacciones de la base de datos MySQL usando tus herramientas "
    "y luego transforma los datos para dejarlos listos para entrenar un modelo de ML."
)

print("=======================================================================")
print(" SUGERENCIAS DE INSTRUCCIONES PARA EL MODELO:")
print("-----------------------------------------------------------------------")
print(f" [Defecto]     -> {prompt_predeterminado}")
print(" [Sugerencia 1] -> 'Extrae solo 10 transacciones de MySQL sin transformar nada.'")
print(" [Sugerencia 2] -> 'Aplica la transformación de ML sobre el archivo temp_dataset.csv ya existente.'")
print("=======================================================================\n")

user_input = input("✍️ Instrucción (ENTER para usar la predeterminada): ").strip()
prompt_activo = user_input if user_input else prompt_predeterminado

print(f"\n✅ Prompt activo: '{prompt_activo}'")

---

## 🚀 Paso 9: Orquestador del Pipeline MCP con Visualización JSON-RPC
Este es el núcleo del sistema. Conecta el Servidor MCP local, descubre las herramientas, consulta al LLM y ejecuta las acciones locales, todo mostrando el flujo de mensajes JSON-RPC con tarjetas de diseño premium.

In [ ]:
SYSTEM_PROMPT = (
    "Eres un agente científico de datos. Tu tarea es usar tus herramientas locales "
    "para extraer y transformar datos de MySQL. Si una herramienta retorna un error "
    "o aviso (como modo offline), analiza el mensaje y decide si continuar con el "
    "siguiente paso, reintentar con argumentos corregidos, o detenerte explicando "
    "el fallo. Responde de forma concisa en español."
)

# ── Constantes de reintentos ──────────────────────────────────────────────────
MAX_RETRIES = 4          # Máximo de reintentos por llamada a la API del LLM
BACKOFF_BASE_SECONDS = 2 # Base del backoff exponencial (2, 4, 8, 16 seg)
MAX_REACT_TURNS = 10     # Límite de seguridad para el bucle ReAct


async def _send_with_retries(send_coro_factory, descripcion="LLM"):
    """Ejecuta una corrutina de envío al LLM con reintentos y backoff exponencial
    para errores 429 (Rate Limit) y 500 (Server Error).
    
    Args:
        send_coro_factory: Callable sin argumentos que retorna una nueva corrutina
                           cada vez que se invoca (necesario porque las corrutinas
                           no se pueden reutilizar tras un await).
        descripcion: Etiqueta para los mensajes de log.
    """
    for intento in range(1, MAX_RETRIES + 1):
        try:
            return await send_coro_factory()
        except Exception as e:
            error_str = str(e).lower()
            codigo = getattr(e, 'status_code', None) or getattr(e, 'code', None)
            es_rate_limit = (codigo == 429) or ('429' in error_str) or ('rate' in error_str and 'limit' in error_str)
            es_server_error = (codigo == 500) or ('500' in error_str) or ('internal server error' in error_str)

            if (es_rate_limit or es_server_error) and intento < MAX_RETRIES:
                wait = BACKOFF_BASE_SECONDS ** intento
                print(f"⚠️ [{descripcion}] Error {'429 Rate Limit' if es_rate_limit else '500 Server'} "
                      f"en intento {intento}/{MAX_RETRIES}. Reintentando en {wait}s...")
                await asyncio.sleep(wait)
            else:
                raise


async def correr_pipeline_mcp(prompt, proveedor, modelo):
    """Orquesta el pipeline completo MCP con un bucle ReAct multi-turno real,
    reintentos con backoff exponencial, manejo de errores de herramientas,
    y visualización JSON-RPC."""
    
    respuesta_final = None
    
    print(f"🔌 [MCP] Conectando al Servidor local vía stdio...")
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # ── A. HANDSHAKE: SOLICITAR CATÁLOGO DE HERRAMIENTAS ──────────────
            mostrar_log_json_rpc(
                "CLIENTE_ENVIANDO", "tools/list",
                {"jsonrpc": "2.0", "method": "tools/list", "id": 1}
            )
            tools_response = await session.list_tools()

            # ── B. MOSTRAR CATÁLOGO DEVUELTO POR EL SERVIDOR ──────────────────
            mostrar_log_json_rpc(
                "SERVIDOR_RESPONDIENDO", "tools/list (RESPONSE)",
                {
                    "jsonrpc": "2.0",
                    "result": {"tools": [
                        {"name": t.name, "description": t.description}
                        for t in tools_response.tools
                    ]},
                    "id": 1
                }
            )

            # ── C. INFERENCIA CON EL MODELO ELEGIDO ───────────────────────────
            print(f"\n🧠 Consultando al modelo '{modelo}' ({proveedor.upper()})...")
            
            idx_call = 10

            if proveedor == "gemini":
                # Convertir y LIMPIAR esquemas para el SDK de Google
                gemini_functions = [
                    {
                        "name": t.name,
                        "description": t.description,
                        "parameters": limpiar_schema_para_gemini(t.inputSchema)
                    }
                    for t in tools_response.tools
                ]
                genai.configure(api_key=gemini_key)
                model = genai.GenerativeModel(
                    model_name=modelo,
                    tools=gemini_functions,
                    system_instruction=SYSTEM_PROMPT
                )
                
                chat = model.start_chat(history=[])
                prompt_actual = prompt
                
                from google.generativeai import protos
                
                for turno in range(MAX_REACT_TURNS):
                    response = await _send_with_retries(
                        lambda p=prompt_actual: chat.send_message_async(p),
                        descripcion=f"Gemini turno {turno+1}"
                    )
                    
                    parts = response.candidates[0].content.parts
                    tool_calls = [p.function_call for p in parts if p.function_call.name]
                    
                    if not tool_calls:
                        respuesta_final = response.text
                        break
                    
                    partes_respuesta = []
                    
                    for fn_call in tool_calls:
                        nombre_fn = fn_call.name
                        argumentos = dict(fn_call.args)
                        
                        mostrar_log_json_rpc(
                            "IA_DECIDIENDO", f"function_call → {nombre_fn}",
                            {"model": modelo, "function": nombre_fn, "arguments": argumentos}
                        )
                        mostrar_log_json_rpc(
                            "CLIENTE_ENVIANDO", "tools/call (REQUEST)",
                            {"jsonrpc": "2.0", "method": "tools/call",
                             "params": {"name": nombre_fn, "arguments": argumentos}, "id": idx_call}
                        )
                        
                        resultado = await session.call_tool(nombre_fn, argumentos)
                        resultado_texto = resultado.content[0].text
                        
                        mostrar_log_json_rpc(
                            "SERVIDOR_RESPONDIENDO", "tools/call (RESPONSE)",
                            {"jsonrpc": "2.0",
                             "result": {"content": [{"type": "text", "text": resultado_texto}]},
                             "id": idx_call}
                        )
                        
                        # Construir respuesta de herramienta con protos verificados
                        partes_respuesta.append(
                            protos.Part(
                                function_response=protos.FunctionResponse(
                                    name=nombre_fn,
                                    response={"result": resultado_texto}
                                )
                            )
                        )
                        idx_call += 1
                    
                    # Retransmitir resultados (incluidos errores/avisos) al LLM
                    # para que decida el siguiente paso del bucle ReAct
                    prompt_actual = partes_respuesta
                else:
                    respuesta_final = (
                        f"[ADVERTENCIA] El bucle ReAct alcanzó el límite de {MAX_REACT_TURNS} turnos "
                        f"sin que el modelo emitiera una respuesta final."
                    )

            else:  # openrouter (SDK OpenAI)
                llm_client = AsyncOpenAI(
                    base_url="https://openrouter.ai/api/v1",
                    api_key=openrouter_key
                )
                openrouter_tools = [
                    {"type": "function", "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema
                    }}
                    for t in tools_response.tools
                ]
                
                messages = [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt}
                ]
                
                for turno in range(MAX_REACT_TURNS):
                    response = await _send_with_retries(
                        lambda: llm_client.chat.completions.create(
                            model=modelo,
                            messages=messages,
                            tools=openrouter_tools
                        ),
                        descripcion=f"OpenRouter turno {turno+1}"
                    )
                    mensaje = response.choices[0].message
                    messages.append(mensaje)
                    
                    if not mensaje.tool_calls:
                        respuesta_final = mensaje.content
                        break
                    
                    for tool_call in mensaje.tool_calls:
                        nombre_fn = tool_call.function.name
                        argumentos = json.loads(tool_call.function.arguments)
                        
                        mostrar_log_json_rpc(
                            "IA_DECIDIENDO", f"tool_call → {nombre_fn}",
                            {"tool_call_id": tool_call.id, "model": modelo,
                             "function": nombre_fn, "arguments": argumentos}
                        )
                        mostrar_log_json_rpc(
                            "CLIENTE_ENVIANDO", "tools/call (REQUEST)",
                            {"jsonrpc": "2.0", "method": "tools/call",
                             "params": {"name": nombre_fn, "arguments": argumentos}, "id": idx_call}
                        )
                        
                        resultado = await session.call_tool(nombre_fn, argumentos)
                        resultado_texto = resultado.content[0].text
                        
                        mostrar_log_json_rpc(
                            "SERVIDOR_RESPONDIENDO", "tools/call (RESPONSE)",
                            {"jsonrpc": "2.0",
                             "result": {"content": [{"type": "text", "text": resultado_texto}]},
                             "id": idx_call}
                        )
                        
                        # El resultado (sea éxito, error o aviso) se retransmite
                        # íntegramente al LLM para que tome la decisión de continuar,
                        # corregir argumentos o detenerse explicando el fallo
                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call.id,
                            "name": nombre_fn,
                            "content": resultado_texto
                        })
                        idx_call += 1
                else:
                    respuesta_final = (
                        f"[ADVERTENCIA] El bucle ReAct alcanzó el límite de {MAX_REACT_TURNS} turnos "
                        f"sin que el modelo emitiera una respuesta final."
                    )

    return respuesta_final

print("✅ Orquestador cargado y listo.")

---

## 🟢 Paso 10: Ejecutar el Pipeline en Vivo
Lanzamos la ejecución. Verás los paquetes JSON-RPC neón en tiempo real y al final la respuesta en lenguaje natural del modelo seleccionado.

In [ ]:
respuesta = await correr_pipeline_mcp(prompt_activo, ia_proveedor, modelo_activo)

---

## 📋 Paso 11: Respuesta Final del Modelo en Lenguaje Natural
Aquí se muestra el texto final que el modelo de IA devolvió al usuario como resultado de la orquestación completa.

In [ ]:
display(HTML(f"""
<div style="border:2px solid #a855f7;background:rgba(168,85,247,0.07);padding:20px;
            border-radius:10px;margin:12px 0;font-family:sans-serif;
            box-shadow:0 4px 20px rgba(168,85,247,0.2);">
  <div style="font-weight:bold;color:#a855f7;font-size:1.2em;margin-bottom:10px;">
    💬 RESPUESTA FINAL DEL MODELO ({modelo_activo.upper()})
  </div>
  <div style="color:#e2e8f0;font-size:1em;line-height:1.6;">
    {respuesta}
  </div>
</div>
"""))